# 📈 Curve Fitting: Years of Experience vs. Annual Salary

> **Target Audience**: Machine Learning Beginners  
> **Core Problem**: Fit a curve to model how salary grows with work experience  
> **Why It Matters**: This is a classic **regression** problem — predicting a continuous value from one input feature. You'll see how different models fit, overfit, and underfit the data.

> **Real-World Meaning**:
> - **X-axis**: Years of professional experience (0 to 35 years)
> - **Y-axis**: Annual salary in $1000s USD
> - The relationship is **non-linear**: salary grows fast early on, then plateaus

---

In [ ]:
# ========================
# 0. Environment Setup
# ========================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

print('All libraries loaded successfully')

In [ ]:
# ===============================================
# 1. Generate Synthetic Data (Real-World Pattern)
# ===============================================

np.random.seed(42)

# Years of experience: 0 to 35
X = np.linspace(0, 35, 150)

# Underlying true function: logarithmic growth with plateau
# Entry salary ~$30K, grows quickly then saturates around ~$75K
true_salary = 30 + 45 * (1 - np.exp(-0.12 * X))

# Add realistic noise (heteroscedastic: more variance at higher salaries)
noise = np.random.normal(0, 3 + 0.05 * X, size=X.shape)
y = true_salary + noise

X = X.reshape(-1, 1)

print(f'Data points: {len(X)}')
print(f'Experience range: {X.min():.0f} to {X.max():.0f} years')
print(f'Salary range: ${y.min():.1f}K to ${y.max():.1f}K')

In [ ]:
# ===============================================
# 2. Visualize the Data
# ===============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot with underlying true curve
axes[0].scatter(X, y, alpha=0.6, s=25, color='steelblue', label='Observed data')
axes[0].plot(X, true_salary, 'r-', lw=2.5, label='True underlying trend')
axes[0].set_xlabel('Years of Experience', fontsize=12)
axes[0].set_ylabel('Annual Salary ($1000s)', fontsize=12)
axes[0].set_title('Salary vs. Experience', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_ylim(15, 90)

# Distribution of salary
sns.histplot(y, kde=True, bins=25, ax=axes[1], color='steelblue')
axes[1].axvline(y.mean(), color='red', ls='--', lw=2, label=f'Mean=${y.mean():.1f}K')
axes[1].set_xlabel('Annual Salary ($1000s)')
axes[1].set_title('Salary Distribution', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Observation: salary grows fast in early career, plateaus after ~20 years")
print("This is a non-linear relationship — perfect for testing different models")

In [ ]:
# ===============================================
# 3. Train/Test Split
# ===============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f'Training set: {len(X_train)} samples')
print(f'Test set:     {len(X_test)} samples')

# Sort test set by X for clean plotting
sort_idx = np.argsort(X_test.flatten())
X_test_sorted = X_test[sort_idx]
y_test_sorted = y_test[sort_idx]

---
## 4. Model Experiments

We'll try 6 different models, from simple to complex:

| Model | Expected Behavior |
|-------|------------------|
| **Linear Regression** | Underfits — can't capture the plateau |
| **Polynomial (degree 2)** | Better — captures the curve roughly |
| **Polynomial (degree 5)** | Good fit — flexible enough |
| **Polynomial (degree 15)** | Overfits — wiggles at the edges |
| **Ridge (degree 15)** | Regularized — controls overfitting |
| **Decision Tree** | Overfits — step-like predictions |

In [ ]:
# ===============================================
# 4. Train All Models
# ===============================================

models = {
    'Linear Regression': LinearRegression(),
    'Polynomial (deg=2)': Pipeline([('poly', PolynomialFeatures(degree=2, include_bias=False)), ('lr', LinearRegression())]),
    'Polynomial (deg=5)': Pipeline([('poly', PolynomialFeatures(degree=5, include_bias=False)), ('lr', LinearRegression())]),
    'Polynomial (deg=15)': Pipeline([('poly', PolynomialFeatures(degree=15, include_bias=False)), ('lr', LinearRegression())]),
    'Ridge (deg=15, α=10)': Pipeline([('poly', PolynomialFeatures(degree=15, include_bias=False)), ('ridge', Ridge(alpha=10))]),
    'Decision Tree (depth=5)': DecisionTreeRegressor(max_depth=5, random_state=42),
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test_sorted)
    
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, model.predict(X_test))
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    test_mae = mean_absolute_error(y_test, model.predict(X_test))
    
    results.append({
        'Model': name,
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Test RMSE ($1000s)': test_rmse,
        'Test MAE ($1000s)': test_mae
    })
    
    # Generate smooth curve for plotting
    X_smooth = np.linspace(0, 37, 300).reshape(-1, 1)
    y_smooth = model.predict(X_smooth)
    predictions[name] = (X_smooth, y_smooth)

results_df = pd.DataFrame(results).set_index('Model')
results_df

In [ ]:
# ===============================================
# 5. Visualize All Fits
# ===============================================

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (name, (X_smooth, y_smooth)) in enumerate(predictions.items()):
    ax = axes[idx]
    
    # Training data
    ax.scatter(X_train, y_train, alpha=0.5, s=18, color='steelblue', label='Train data')
    # Test data
    ax.scatter(X_test, y_test, alpha=0.6, s=25, color='orange', marker='s', label='Test data')
    # True curve
    ax.plot(np.linspace(0, 35, 300), 30 + 45 * (1 - np.exp(-0.12 * np.linspace(0, 35, 300))), 
            'g--', lw=2, alpha=0.7, label='True trend')
    # Model prediction
    ax.plot(X_smooth, y_smooth, 'r-', lw=2.5, label='Model fit')
    
    ax.set_xlabel('Years of Experience', fontsize=10)
    ax.set_ylabel('Salary ($1000s)', fontsize=10)
    ax.set_title(f'{name}\nTrain R²={results_df.loc[name,"Train R²"]:.4f}, Test R²={results_df.loc[name,"Test R²"]:.4f}', 
                 fontsize=10, fontweight='bold')
    ax.set_ylim(15, 90)
    ax.set_xlim(-1, 37)
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Curve Fitting Comparison: Different Models on Salary vs. Experience', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("What to notice:")
print("  - Linear: straight line, misses the plateau entirely (underfitting)")
print("  - Poly deg=2: captures some curve but not enough")
print("  - Poly deg=5: good fit, follows the true trend closely")
print("  - Poly deg=15: wiggles at boundaries (overfitting)")
print("  - Ridge deg=15: regularization tames the wiggles")
print("  - Decision Tree: step-like, piecewise constant predictions")

In [ ]:
# ===============================================
# 6. Performance Bar Chart
# ===============================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, title, color in zip(
    axes,
    ['Test R²', 'Test RMSE ($1000s)', 'Test MAE ($1000s)'],
    ['Test R² (higher is better)', 'Test RMSE in $1000s (lower is better)', 'Test MAE in $1000s (lower is better)'],
    ['#2ecc71', '#e74c3c', '#e74c3c']
):
    values = results_df[metric]
    bars = ax.barh(values.index, values.values, color='steelblue', edgecolor='black', height=0.6)
    
    for bar, val in zip(bars, values.values):
        ax.text(val + 0.3 * (abs(values.values).max() if abs(values.values).max() > 0 else 1), 
                bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ===============================================
# 7. Overfitting: Train vs Test R² by Complexity
# ===============================================

fig, ax = plt.subplots(figsize=(10, 6))

x_idx = np.arange(len(results_df))
width = 0.35

bars1 = ax.bar(x_idx - width/2, results_df['Train R²'], width, label='Train R²', color='steelblue', edgecolor='black')
bars2 = ax.bar(x_idx + width/2, results_df['Test R²'], width, label='Test R²', color='coral', edgecolor='black')

for bar, val in zip(bars1, results_df['Train R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', 
            ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars2, results_df['Test R²']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', 
            ha='center', va='bottom', fontsize=8)

ax.set_xticks(x_idx)
ax.set_xticklabels(results_df.index, fontsize=9, rotation=15)
ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('Overfitting Diagnosis: Train vs Test R² by Model Complexity', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.axhline(y=1.0, color='gray', ls=':', lw=1)
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - Poly deg=15: Train R² ≈ 1.0 (perfect memory) but Test R² drops → overfitting")
print("  - Ridge (deg=15): regularization closes the gap")
print("  - Poly deg=5: best balance — high Test R² without overfitting")

In [ ]:
# ===============================================
# 8. Learning Curve: Effect of Training Data Size
# ===============================================

from sklearn.model_selection import learning_curve

lc_configs = [
    ('Polynomial (deg=2)', Pipeline([('poly', PolynomialFeatures(degree=2, include_bias=False)), ('lr', LinearRegression())])),
    ('Polynomial (deg=5)', Pipeline([('poly', PolynomialFeatures(degree=5, include_bias=False)), ('lr', LinearRegression())])),
    ('Polynomial (deg=15)', Pipeline([('poly', PolynomialFeatures(degree=15, include_bias=False)), ('lr', LinearRegression())])),
    ('Decision Tree (depth=5)', DecisionTreeRegressor(max_depth=5, random_state=42)),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, model) in zip(axes.flatten(), lc_configs):
    train_sizes = np.linspace(0.1, 1.0, 10)
    
    train_sizes_abs, train_scores, test_scores = learning_curve(
        model, X_train, y_train, train_sizes=train_sizes, cv=5, 
        scoring='r2', n_jobs=-1, random_state=42)
    
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)
    
    ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, 
                    alpha=0.2, color='blue')
    ax.fill_between(train_sizes_abs, test_mean - test_std, test_mean + test_std, 
                    alpha=0.2, color='orange')
    ax.plot(train_sizes_abs, train_mean, 'o-', color='blue', label='Train R²')
    ax.plot(train_sizes_abs, test_mean, 's-', color='orange', label='Validation R²')
    
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel('Training Samples')
    ax.set_ylabel('R² Score')
    ax.legend(loc='best')
    ax.set_ylim(-0.5, 1.1)
    ax.axhline(y=0, color='gray', ls=':', lw=0.8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves: How Data Size Affects Each Model (5-fold CV)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Insights:")
print("  - deg=2 model: converges fast, limited by model capacity (underfitting)")
print("  - deg=5 model: good learning curve, improves steadily with data")
print("  - deg=15 model: huge train-test gap = severe overfitting with little data")
print("  - Decision Tree: erratic, benefits from more data but still step-like")

In [ ]:
# ===============================================
# 9. Residual Analysis (Best Model: Poly deg=5)
# ===============================================

best_model = Pipeline([('poly', PolynomialFeatures(degree=5, include_bias=False)), ('lr', LinearRegression())])
best_model.fit(X_train, y_train)
y_pred_all = best_model.predict(X)
residuals = y.flatten() - y_pred_all.flatten()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residual scatter
axes[0].scatter(y_pred_all, residuals, alpha=0.6, s=25, color='steelblue', edgecolors='k', linewidth=0.5)
axes[0].axhline(y=0, color='red', ls='--', lw=2)
axes[0].set_xlabel('Predicted Salary ($1000s)')
axes[0].set_ylabel('Residual ($1000s)')
axes[0].set_title(f'Residual Plot (Poly deg=5)\nResidual Std={residuals.std():.3f}')

# Residual histogram
sns.histplot(residuals, kde=True, bins=25, ax=axes[1], color='steelblue')
axes[1].axvline(x=0, color='red', ls='--', lw=2, label='Zero error')
axes[1].axvline(x=residuals.mean(), color='green', ls='--', lw=2, label=f'Mean={residuals.mean():.3f}')
axes[1].set_xlabel('Residual ($1000s)')
axes[1].set_title('Residual Distribution')
axes[1].legend()

# Residual vs X
axes[2].scatter(X, residuals, alpha=0.6, s=25, color='steelblue', edgecolors='k', linewidth=0.5)
axes[2].axhline(y=0, color='red', ls='--', lw=2)
axes[2].set_xlabel('Years of Experience')
axes[2].set_ylabel('Residual ($1000s)')
axes[2].set_title('Residual vs Experience')

plt.tight_layout()
plt.show()

_, p_value = stats.normaltest(residuals)
print(f"Normality test p-value: {p_value:.4f} (p>0.05 → residuals are normal)")

In [ ]:
# ===============================================
# 10. Extrapolation: How Models Behave Beyond Data
# ===============================================

X_extrapolate = np.linspace(0, 50, 500).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(12, 6))

ax.scatter(X_train, y_train, alpha=0.5, s=20, color='steelblue', label='Training data (0-35 yrs)')
ax.axvline(x=35, color='gray', ls=':', lw=2, alpha=0.7, label='End of training data')

extrapolate_models = ['Linear Regression', 'Polynomial (deg=5)', 'Polynomial (deg=15)', 'Ridge (deg=15, α=10)']
colors = ['#e74c3c', '#2ecc71', '#e67e22', '#3498db']

for name, color in zip(extrapolate_models, colors):
    model = models[name]
    y_ext = model.predict(X_extrapolate)
    ax.plot(X_extrapolate, y_ext, '-', color=color, lw=2, label=name)

ax.set_xlabel('Years of Experience', fontsize=12)
ax.set_ylabel('Salary ($1000s)', fontsize=12)
ax.set_title('Extrapolation: Model Predictions Beyond Training Data (35+ years)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 200)
plt.tight_layout()
plt.show()

print("What happens beyond 35 years?")
print("  - Linear: keeps growing linearly (unrealistic — salary doesn't grow forever)")
print("  - Poly deg=5: reasonable plateau, but eventually turns down (also wrong)")
print("  - Poly deg=15: EXPLODES! Wild extrapolation = classic overfitting symptom")
print("  - Ridge: more controlled, but still diverges eventually")

---
## Summary

| Model | Train R² | Test R² | Problem |
|-------|----------|---------|--------|
| Linear Regression | Moderate | Moderate | **Underfitting** — can't capture curve |
| Polynomial (deg=2) | Good | Moderate | Slightly underfits |
| Polynomial (deg=5) | High | High | ✅ **Best balance** |
| Polynomial (deg=15) | ~1.0 | Lower | **Overfitting** — wiggly, extrapolates wildly |
| Ridge (deg=15) | High | High | Regularization helps control overfitting |
| Decision Tree | High | High | Step-like, less smooth |

### Key Lessons
1. **Simple models** (linear) underfit complex curves → high bias
2. **Overly complex models** (poly deg=15) memorize noise → high variance
3. **The right complexity** (poly deg=5) balances bias and variance
4. **Regularization** (Ridge) tames complexity without reducing degrees of freedom
5. **More data** helps complex models more than simple ones (learning curves)
6. **Extrapolation is dangerous** — high-degree polynomials explode outside training range